# Fine-tune VietEmbed RAG with 2 GPUs (Kaggle DDP)

Before running, enable a two-GPU accelerator in Kaggle and attach the dataset containing `general_export.db`.

In [ ]:
import torch

gpu_count = torch.cuda.device_count()
print(f"Available GPUs: {gpu_count}")
for index in range(gpu_count):
    print(f"cuda:{index}: {torch.cuda.get_device_name(index)}")

if gpu_count != 2:
    raise RuntimeError("Enable a 2-GPU accelerator in Kaggle before training.")

In [ ]:
%%writefile /kaggle/working/finetune_general_ddp.py
import math
import os
import sqlite3

import torch
from datasets import Features, IterableDataset, Value
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)

DB_PATH = "/kaggle/input/datasets/nhminh107/data-vietembed-rag/general_export.db"
MODEL_NAME = "intfloat/multilingual-e5-base"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "
OUTPUT_DIR = "/kaggle/working/VietEmbed-RAG"
BATCH_SIZE = 32
EPOCHS = 2
GRADIENT_ACCUMULATION_STEPS = 1


def get_general_data(batch_size: int = 10_000):
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    cursor = connection.execute(
        """
        SELECT anchor, positive
        FROM general
        WHERE anchor IS NOT NULL
          AND positive IS NOT NULL
          AND source <> "Wikipedia vi"
        ORDER BY data_id ASC
        """
    )

    try:
        while rows := cursor.fetchmany(batch_size):
            for row in rows:
                yield {
                    "anchor": f"{QUERY_PREFIX}{row['anchor']}",
                    "positive": f"{PASSAGE_PREFIX}{row['positive']}",
                }
    finally:
        cursor.close()
        connection.close()


def count_records() -> int:
    connection = sqlite3.connect(DB_PATH)
    try:
        return connection.execute(
            """
            SELECT COUNT(*)
            FROM general
            WHERE anchor IS NOT NULL
              AND positive IS NOT NULL
            """
        ).fetchone()[0]
    finally:
        connection.close()


def main() -> None:
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    if torch.cuda.device_count() < world_size:
        raise RuntimeError("torchrun requested more GPUs than are available.")

    num_records = count_records()
    steps_per_epoch = num_records // (
        BATCH_SIZE * world_size * GRADIENT_ACCUMULATION_STEPS
    )
    max_steps = steps_per_epoch * EPOCHS
    if rank == 0:
        print(f"Records: {num_records:,}")
        print(f"World size: {world_size}")
        print(f"Effective batch size: {BATCH_SIZE * world_size * GRADIENT_ACCUMULATION_STEPS}")
        print(f"Steps per epoch: {steps_per_epoch:,}")
        print(f"Max steps: {max_steps:,}")

    train_dataset = IterableDataset.from_generator(
        get_general_data,
        features=Features({"anchor": Value("string"), "positive": Value("string")}),
        gen_kwargs={"batch_size": 10_000},
    )

    model = SentenceTransformer(MODEL_NAME)
    loss = losses.MultipleNegativesRankingLoss(model)
    arguments = SentenceTransformerTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        max_steps=max_steps,
        learning_rate=2e-5,
        warmup_steps=int(max_steps * 0.1),
        fp16=True,
        dataloader_num_workers=0,
        dataloader_drop_last=True,
        accelerator_config={"dispatch_batches": False},
        ddp_find_unused_parameters=True,
        logging_steps=100,
        save_strategy="steps",
        save_steps=5_000,
        save_total_limit=2,
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=arguments,
        train_dataset=train_dataset,
        loss=loss,
    )
    trainer.train()
    if trainer.is_world_process_zero():
        trainer.save_model(OUTPUT_DIR)


if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/finetune_general_ddp.py

In [ ]:
from pathlib import Path
import shutil

model_dir = Path("/kaggle/working/VietEmbed-RAG")
if not (model_dir / "modules.json").is_file():
    raise RuntimeError("Model is missing. Check the torchrun cell for the training error.")

archive_path = shutil.make_archive(
    str(model_dir),
    "zip",
    root_dir=model_dir.parent,
    base_dir=model_dir.name,
)

print(f"Final model directory: {model_dir}")
print(f"Downloadable archive: {archive_path}")
print(f"Model files: {len(list(model_dir.rglob('*')))}")